# Case file: 2700 Shattuck (359 units)

**For reporters.** The ZAB approved this mixed-use project in July 2026 after a debate over
waiving an affordable-housing requirement
([Daily Cal](https://www.dailycal.org/news/city/housing/berkeley-zoning-board-approves-shattuck-mixed-use-development-debates-waiving-affordable-housing-requirement/article_3c5c9878-a6f8-4cb6-adb0-53230ef59bb5.html)).
This notebook pulls the project's full history — and the citywide context for the two big
questions — from **berkeleybuild.com's public database**, computing every number in front of you.
Run each cell (Runtime → Run all). Edit any query; it's your copy.

*The record: an independent reconstruction from Berkeley's own permit data (every permit
2015–2026), audited row-by-row against the city's state filing (±130 units of ~4,200, every
difference named — [the audit](https://berkeleybuild.com/housing-audit.html)).*

In [1]:
# Load the public database (fetched from berkeleybuild.com; ~4 MB)
import sqlite3, urllib.request
from pathlib import Path
DB = Path('berkeley_housing_v2_public.db')
LOCAL = Path('docs/data/berkeley_housing_v2_public.db')   # repo checkout fallback
if LOCAL.exists(): DB = LOCAL
elif not DB.exists():
    urllib.request.urlretrieve('https://berkeleybuild.com/data/berkeley_housing_v2_public.db', DB)
con = sqlite3.connect(f'file:{DB}?mode=ro', uri=True)
import pandas as pd
print('projects on record:', con.execute('SELECT COUNT(*) FROM v_projects_flat').fetchone()[0])

projects on record: 895


## 1 · The project's own clock
Every recorded event for 2700 Shattuck, oldest first.

In [2]:
# auto-setup — safe to re-run; downloads only if needed (cells can run in ANY order)
import sqlite3, urllib.request; from pathlib import Path
import pandas as pd
if 'con' not in globals():
    _db = Path('docs/data/berkeley_housing_v2_public.db')
    if not _db.exists():
        _db = Path('berkeley_housing_v2_public.db')
        if not _db.exists():
            urllib.request.urlretrieve('https://berkeleybuild.com/data/berkeley_housing_v2_public.db', _db)
    con = sqlite3.connect(f'file:{_db}?mode=ro', uri=True)
timeline = pd.read_sql("""
    SELECT substr(e.event_date,1,10) AS date, t.code AS event, COALESCE(e.summary,'') AS detail
    FROM project_events e
    JOIN vocabulary_event_types t ON t.id=e.event_type_id
    JOIN v_projects_flat f ON f.project_id=e.project_id
    WHERE f.address_display LIKE '2700 SHATTUCK%' ORDER BY 1""", con)
timeline

,date,event,detail
0,2024-06-13,application_submitted,
1,2024-06-13,comments_issued,Incomplete Pending Applicant
2,2024-06-13,comments_issued,Incomplete Pending Applicant
3,2024-10-10,application_complete,
4,2024-10-10,application_complete,Application Complete
5,2024-10-10,application_complete,Application Complete
6,2024-11-07,comments_issued,Corrections - Pending Applicant
7,2024-11-07,comments_issued,Corrections - Pending Applicant
8,2026-04-01,entitlement_approved,"Entitlement: DRCP2024-0006 design review ""Appr..."
9,2026-07-09,entitlement_approved,"Entitlement: ZP2024-0058 ""ZAB Approved 07/09/2..."


**How to read it:** SB 330 pre-application Nov 2023 → full application May/Jun 2024 → marked
*Incomplete* the same day (the intake ping-pong) → **deemed complete 2024-10-10** (the statutory
clock starts) → ZAB approval July 2026. That's ~**21 months in consideration** — the next cell
puts that on the citywide curve.

In [3]:
# auto-setup — safe to re-run; downloads only if needed (cells can run in ANY order)
import sqlite3, urllib.request; from pathlib import Path
import pandas as pd
if 'con' not in globals():
    _db = Path('docs/data/berkeley_housing_v2_public.db')
    if not _db.exists():
        _db = Path('berkeley_housing_v2_public.db')
        if not _db.exists():
            urllib.request.urlretrieve('https://berkeleybuild.com/data/berkeley_housing_v2_public.db', _db)
    con = sqlite3.connect(f'file:{_db}?mode=ro', uri=True)
clock = pd.read_sql("""
    SELECT address_display, total_units,
           CAST(julianday(entitled_date)-julianday(filed_date) AS INT) AS days_to_entitle
    FROM v_projects_flat
    WHERE total_units>=50 AND filed_date IS NOT NULL AND entitled_date IS NOT NULL
      AND julianday(entitled_date) > julianday(filed_date)""", con)
print(f"major projects with both dates: {len(clock)}")
print(f"median application->entitlement: {clock.days_to_entitle.median():.0f} days")
print(f"2700 Shattuck, application (2024-06) -> approval (2026-07): ~760 days and counting to a permit")
clock.sort_values('days_to_entitle', ascending=False).head(10)

major projects with both dates: 34
median application->entitlement: 527 days
2700 Shattuck, application (2024-06) -> approval (2026-07): ~760 days and counting to a permit


,address_display,total_units,days_to_entitle
31,1752 SHATTUCK Ave,72,1485
21,2136 SAN PABLO Ave,125,1161
6,3000 SHATTUCK Ave,166,1035
28,2001 ASHBY Ave,87,846
26,3030 TELEGRAPH Ave,144,803
22,2016 ASHBY Ave,50,784
13,1974 SHATTUCK Ave,599,770
2,2700 SHATTUCK Ave,359,756
18,2427 San Pablo,78,712
9,2372 ELLSWORTH St,63,687


## 2 · The affordability-waiver context
The debate was about the **State Density Bonus** (Gov. Code 65915): extra height/units in exchange
for below-market units, with waivers of local standards. Who else uses it in Berkeley:

In [4]:
# auto-setup — safe to re-run; downloads only if needed (cells can run in ANY order)
import sqlite3, urllib.request; from pathlib import Path
import pandas as pd
if 'con' not in globals():
    _db = Path('docs/data/berkeley_housing_v2_public.db')
    if not _db.exists():
        _db = Path('berkeley_housing_v2_public.db')
        if not _db.exists():
            urllib.request.urlretrieve('https://berkeleybuild.com/data/berkeley_housing_v2_public.db', _db)
    con = sqlite3.connect(f'file:{_db}?mode=ro', uri=True)
db = pd.read_sql("""
    SELECT f.address_display, f.total_units, f.status_code, substr(f.entitled_date,1,10) AS entitled
    FROM project_classifications pc
    JOIN vocabulary_classification_types v ON pc.classification_type_id=v.id
    JOIN v_projects_flat f ON f.project_id=pc.project_id
    WHERE v.code='density_bonus' ORDER BY f.total_units DESC""", con)
print(f"density-bonus projects in the curated record: {len(db)} "
      "(text-mention floor across all planning filings: ~134 — see berkeleybuild.com)")
db.head(15)

density-bonus projects in the curated record: 54 (text-mention floor across all planning filings: ~134 — see berkeleybuild.com)


,address_display,total_units,status_code,entitled
0,Ashby BART,618,pre_application,None
1,1974 SHATTUCK Ave,599,entitled,2025-06-03
2,2190 SHATTUCK Ave,452,in_review,None
3,2276 SHATTUCK Ave,336,entitled,2025-12-04
4,2274 SHATTUCK Ave,299,entitled,2025-04-22
5,1914 FIFTH St,257,in_review,None
6,2920 SHATTUCK Ave,242,in_review,None
7,2029 UNIVERSITY Ave,240,entitled,2025-11-13
8,2601 SAN PABLO Ave,223,in_review,None
9,1899 OXFORD St,212,entitled,2026-02-11


## 3 · What approval means statistically — the waiting room
Approval is not a building. Entitled projects next need a **building permit**, and that queue is
where Berkeley's pipeline actually narrows:

In [5]:
# auto-setup — safe to re-run; downloads only if needed (cells can run in ANY order)
import sqlite3, urllib.request; from pathlib import Path
import pandas as pd
if 'con' not in globals():
    _db = Path('docs/data/berkeley_housing_v2_public.db')
    if not _db.exists():
        _db = Path('berkeley_housing_v2_public.db')
        if not _db.exists():
            urllib.request.urlretrieve('https://berkeleybuild.com/data/berkeley_housing_v2_public.db', _db)
    con = sqlite3.connect(f'file:{_db}?mode=ro', uri=True)
wr = pd.read_sql("""
    SELECT address_display, total_units, substr(entitled_date,1,10) AS entitled
    FROM v_projects_flat
    WHERE entitled_date IS NOT NULL AND bp_issued_date IS NULL AND co_issued_date IS NULL
      AND status_code != 'withdrawn' ORDER BY total_units DESC""", con)
print(f"THE WAITING ROOM today: {len(wr)} entitled projects, {wr.total_units.sum():,.0f} units, no permit")
print("Under BMC 23.404.060 an approval becomes lapse-ELIGIBLE one year after decision unless")
print("exercised — 2700 Shattuck's clock starts at its approval date.")
wr.head(12)

THE WAITING ROOM today: 34 entitled projects, 5,470 units, no permit
Under BMC 23.404.060 an approval becomes lapse-ELIGIBLE one year after decision unless
exercised — 2700 Shattuck's clock starts at its approval date.


,address_display,total_units,entitled
0,1750 SACRAMENTO St,739,2026-03-03
1,1974 SHATTUCK Ave,599,2025-06-03
2,2128 Oxford St,485,2024-10-04
3,2700 SHATTUCK Ave,359,2026-07-09
4,2276 SHATTUCK Ave,336,2025-12-04
5,2274 SHATTUCK Ave,299,2025-04-22
6,2029 UNIVERSITY Ave,240,2025-11-13
7,1899 OXFORD St,212,2026-02-11
8,2100 MILVIA St,205,2025-07-01
9,2131 University Ave,205,2013-07-08


## Cite & dig further
- Cite: *"berkeleybuild.com, an independent reconstruction of Berkeley's housing records from city
  permit data."*
- Browse this database with no code at all: [the reporters page](https://berkeleybuild.com/reporters.html)
  has one-click queries. The method, the audit, and the rebuild-it-yourself curriculum are all public.
- Something you need verified against the primary documents? Ask — that's what the project is for.